# EuroSAT 5-Fold CV

**Week 6 / Task 1**

This notebook runs a stratified **5-fold cross-validation** benchmark on **EuroSAT-MS** and exports:

- `fold metrics CSV`
- `mean±std comparison table`
- `paired t-test p-values`
- `per-class summary table`

The notebook evaluates the local methods already supported in this repo:

- `RGB-CLIP`
- `PCA`
- `NDVI`
- `Tip-Adapter`
- `RS-TransCLIP`
- `Ours`

`DOFA` is exposed through **external CSV templates** because the repo currently does not contain a DOFA model implementation.


In [7]:
import sys
from pathlib import Path

import pandas as pd
import torch

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.experiments.eurosat_5fold_cv import (
    DEFAULT_ALL_METHODS,
    run_eurosat_5fold_cv,
)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
# NOTE: Do NOT call torch.set_grad_enabled(False) globally here.
# The "Ours" pipeline runs test-time optimization with Adam and needs grads.
# Encoding paths already use inference/no_grad internally where appropriate.

print(f"Project root: {PROJECT_ROOT}")
print(f"PyTorch: {torch.__version__}")

Project root: /Users/tienesng06/Desktop/ACIVS_ThayBach
PyTorch: 2.0.1


## Config

Use `METHODS` and `FOLD_IDS` to control runtime:

- Fast smoke test: `METHODS = ["RGB-CLIP", "PCA"]`, `FOLD_IDS = [0]`
- Even faster smoke test: also set `MAX_TRAIN_PER_CLASS = 24`, `MAX_GALLERY_PER_CLASS = 8`
- `MAX_GALLERY_PER_CLASS` caps both held-out splits: the query fold and the gallery fold
- Full local run: keep the default method list and set `FOLD_IDS = None`
- Add `DOFA`: fill the template CSVs printed below, then set `EXTERNAL_METHOD_INPUTS["DOFA"]`


In [8]:
EUROSAT_ROOT = PROJECT_ROOT / "data" / "EuroSAT_MS.local_backup_20260419"
CLIP_CHECKPOINT = PROJECT_ROOT / "checkpoints" / "ViT-B-16.pt"
RESULTS_DIR = PROJECT_ROOT / "results" / "eurosat_5fold_cv"

METHODS = [
    "RGB-CLIP",
    "PCA",
    "NDVI",
    "Tip-Adapter",
    "RS-TransCLIP",
    "Ours",
    # "DOFA",
]

FOLD_IDS = None
NUM_FOLDS = 5
SEED = 42
BATCH_SIZE = 128
NUM_WORKERS = 0
SHOW_PROGRESS = True
# MAX_GALLERY_PER_CLASS caps both held-out splits: query + gallery.
MAX_TRAIN_PER_CLASS = None
MAX_GALLERY_PER_CLASS = None

EXTERNAL_METHOD_INPUTS = {
    # "DOFA": {
    #     "summary_csv": RESULTS_DIR / "external_templates" / "dofa_fold_metrics_template.csv",
    #     "per_query_csv": RESULTS_DIR / "external_templates" / "dofa_per_query_template.csv",
    # }
}

TIP_ADAPTER_ALPHA = 20.0
TIP_ADAPTER_BETA = 5.5
RS_TRANSCLIP_ALPHA = 0.3

OURS_SIGMA = 0.5
OURS_NUM_STEPS = 5
OURS_LR = 0.01
OURS_LAMBDA_M = 0.1
OURS_K = 5

print(f"EuroSAT root exists: {EUROSAT_ROOT.exists()}")
print(f"CLIP checkpoint exists: {CLIP_CHECKPOINT.exists()}")
print(f"Results dir: {RESULTS_DIR}")
print(f"Methods: {METHODS}")
print(f"Folds: {FOLD_IDS if FOLD_IDS is not None else 'all'}")
print(f"Debug caps: train={MAX_TRAIN_PER_CLASS}, heldout={MAX_GALLERY_PER_CLASS}")

EuroSAT root exists: True
CLIP checkpoint exists: True
Results dir: /Users/tienesng06/Desktop/ACIVS_ThayBach/results/eurosat_5fold_cv
Methods: ['RGB-CLIP', 'PCA', 'NDVI', 'Tip-Adapter', 'RS-TransCLIP', 'Ours']
Folds: all
Debug caps: train=None, heldout=None


## Run 5-Fold CV

This cell writes the main artifacts under `results/eurosat_5fold_cv/`.


In [9]:
result = run_eurosat_5fold_cv(
    root=EUROSAT_ROOT,
    clip_checkpoint=CLIP_CHECKPOINT,
    results_dir=RESULTS_DIR,
    methods=METHODS,
    external_method_inputs=EXTERNAL_METHOD_INPUTS,
    num_folds=NUM_FOLDS,
    fold_ids=FOLD_IDS,
    seed=SEED,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    show_progress=SHOW_PROGRESS,
    max_train_per_class=MAX_TRAIN_PER_CLASS,
    max_gallery_per_class=MAX_GALLERY_PER_CLASS,
    tip_adapter_alpha=TIP_ADAPTER_ALPHA,
    tip_adapter_beta=TIP_ADAPTER_BETA,
    rs_transclip_alpha=RS_TRANSCLIP_ALPHA,
    ours_sigma=OURS_SIGMA,
    ours_num_steps=OURS_NUM_STEPS,
    ours_lr=OURS_LR,
    ours_lambda_m=OURS_LAMBDA_M,
    ours_k=OURS_K,
)

print(f"Manifest: {result['manifest_path']}")
print("External templates:")
for method, paths in result["external_templates"].items():
    print(f"  {method}:")
    for name, path in paths.items():
        print(f"    {name}: {path}")

Fold 0 RGB+patch query:   0%|          | 0/43 [00:00<?, ?it/s]

Fold 0 RGB+patch gallery:   0%|          | 0/43 [00:00<?, ?it/s]

Encoding test gallery:   0%|          | 0/127 [00:00<?, ?it/s]

Fold 0 PCA fit:   0%|          | 0/127 [00:00<?, ?it/s]

Fold 0 PCA fit (range):   0%|          | 0/127 [00:00<?, ?it/s]

...

Fold 4 band embeddings query:   0%|          | 0/43 [00:00<?, ?it/s]

Fold 4 band embeddings gallery:   0%|          | 0/43 [00:00<?, ?it/s]

Fold 4 Ours fuse query:   0%|          | 0/5400 [00:00<?, ?it/s]

Fold 4 Ours fuse gallery:   0%|          | 0/5400 [00:00<?, ?it/s]

Manifest: /Users/tienesng06/Desktop/ACIVS_ThayBach/results/eurosat_5fold_cv/eurosat_5fold_manifest.json
External templates:
  DOFA:
    summary_csv: /Users/tienesng06/Desktop/ACIVS_ThayBach/results/eurosat_5fold_cv/external_templates/dofa_fold_metrics_template.csv
    per_query_csv: /Users/tienesng06/Desktop/ACIVS_ThayBach/results/eurosat_5fold_cv/external_templates/dofa_per_query_template.csv


## Split Manifest

Each run uses:

- 1 fold as **query**
- the next fold as **gallery**
- the remaining 3 folds as **train/reference**

This keeps query/gallery disjoint and makes the benchmark a real **image-to-image retrieval** protocol.


In [10]:
result["split_manifest_df"]

,fold_id,query_fold_id,gallery_fold_id,class_idx,class_name,train_count,query_count,gallery_count
0,0,0,1,0,AnnualCrop,1800,600,600
1,0,0,1,1,Forest,1800,600,600
2,0,0,1,2,HerbaceousVegetation,1800,600,600
3,0,0,1,3,Highway,1500,500,500
4,0,0,1,4,Industrial,1500,500,500
5,0,0,1,5,Pasture,1200,400,400
6,0,0,1,6,PermanentCrop,1500,500,500
7,0,0,1,7,Residential,1800,600,600
8,0,0,1,8,River,1500,500,500
9,0,0,1,9,SeaLake,1800,600,600


## Fold Metrics

Raw metrics per fold and per method.


In [11]:
result["fold_metrics_df"]

,method,fold_id,seed,query_fold_id,gallery_fold_id,num_train,num_query,num_gallery,R@1,R@5,R@10,mAP,tip_adapter_alpha,tip_adapter_beta,rs_transclip_alpha,ours_sigma,ours_num_steps,ours_lr,ours_lambda_m,ours_k,avg_query_fusion_ms,avg_query_optimization_ms,avg_gallery_fusion_ms,avg_gallery_optimization_ms
0,RGB-CLIP,0,42,0,1,16200,5400,5400,80.388892,94.629627,97.203702,40.338091,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,RGB-CLIP,1,42,1,2,16200,5400,5400,80.166668,95.722222,97.981483,40.306415,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,RGB-CLIP,2,42,2,3,16200,5400,5400,79.333335,94.537038,97.796297,39.927081,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,RGB-CLIP,3,42,3,4,16200,5400,5400,80.074072,94.759262,97.222221,40.607402,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,RGB-CLIP,4,42,4,0,16200,5400,5400,81.240743,95.074075,97.888887,40.713289,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,PCA,0,42,0,1,16200,5400,5400,83.388889,96.185184,98.037034,42.879319,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,PCA,1,42,1,2,16200,5400,5400,83.796299,96.333331,98.407406,42.732651,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,PCA,2,42,2,3,16200,5400,5400,83.740741,96.351850,98.314816,42.501629,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,PCA,3,42,3,4,16200,5400,5400,83.425927,96.481484,98.388886,42.765113,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,PCA,4,42,4,0,16200,5400,5400,83.759260,95.962965,98.240739,42.839299,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Mean ± Std Comparison

Primary table for the Week 6 summary.


In [12]:
result["comparison_df"]

,method,num_folds,num_query_mean,num_gallery_mean,R@1_mean,R@1_std,R@1_mean_std,R@5_mean,R@5_std,R@5_mean_std,R@10_mean,R@10_std,R@10_mean_std,mAP_mean,mAP_std,mAP_mean_std
0,RGB-CLIP,5,5400.0,5400.0,80.240742,0.685311,80.24±0.69,94.944445,0.479876,94.94±0.48,97.618518,0.376023,97.62±0.38,40.378456,0.306222,40.38±0.31
1,PCA,5,5400.0,5400.0,83.622223,0.197551,83.62±0.20,96.262963,0.197897,96.26±0.20,98.277776,0.149875,98.28±0.15,42.743602,0.147233,42.74±0.15
2,NDVI,5,5400.0,5400.0,83.688889,0.469838,83.69±0.47,96.340741,0.260774,96.34±0.26,98.188888,0.338792,98.19±0.34,45.556153,0.216204,45.56±0.22
3,Tip-Adapter,5,5400.0,5400.0,67.666667,0.613491,67.67±0.61,89.292594,0.504846,89.29±0.50,94.511111,0.313066,94.51±0.31,36.580907,0.280049,36.58±0.28
4,RS-TransCLIP,5,5400.0,5400.0,78.574075,0.905799,78.57±0.91,92.177778,0.612736,92.18±0.61,95.225925,0.428814,95.23±0.43,43.213832,0.313009,43.21±0.31
5,Ours,5,5400.0,5400.0,85.022224,0.677560,85.02±0.68,96.422223,0.214927,96.42±0.21,98.196295,0.237731,98.20±0.24,45.001534,0.294855,45.00±0.29


## Paired t-test

By default this compares every method against `Ours` on the matched fold-level scores.


In [13]:
result["paired_ttest_df"]

,reference_method,compared_method,metric,n_pairs,t_stat,p_value,significant_p_lt_0_05
0,Ours,RGB-CLIP,R@1,5,10.845875,4.100835e-04,True
1,Ours,RGB-CLIP,R@10,5,4.098853,1.486698e-02,True
2,Ours,RGB-CLIP,R@5,5,7.369960,1.806264e-03,True
3,Ours,RGB-CLIP,mAP,5,50.254795,9.382006e-07,True
4,Ours,PCA,R@1,5,6.374367,3.106754e-03,True
5,Ours,PCA,R@10,5,-0.646915,5.529372e-01,False
6,Ours,PCA,R@5,5,0.908241,4.151227e-01,False
7,Ours,PCA,mAP,5,22.455108,2.329005e-05,True
8,Ours,NDVI,R@1,5,3.959277,1.668774e-02,True
9,Ours,NDVI,R@10,5,0.081214,9.391732e-01,False


## Per-Class Table

This table aggregates metrics over the **real query images** of each class across the selected folds.


In [14]:
result["per_class_df"]

,method,query_label,class_name,num_folds,num_queries,mAP_mean,mAP_std,mAP_mean_std,R@1_mean,R@1_std,R@1_mean_std,R@5_mean,R@5_std,R@5_mean_std,R@10_mean,R@10_std,R@10_mean_std
0,RGB-CLIP,0,AnnualCrop,5,3000,40.036042,19.243834,40.04±19.24,83.600000,37.033730,83.60±37.03,96.200000,19.122811,96.20±19.12,98.266667,13.053187,98.27±13.05
1,RGB-CLIP,1,Forest,5,3000,69.750223,23.063212,69.75±23.06,94.566667,22.671205,94.57±22.67,99.100000,9.445620,99.10±9.45,99.466667,7.284681,99.47±7.28
2,RGB-CLIP,2,HerbaceousVegetation,5,3000,44.286981,16.171773,44.29±16.17,89.366667,30.831517,89.37±30.83,98.233333,13.175865,98.23±13.18,99.133333,9.270610,99.13±9.27
3,RGB-CLIP,3,Highway,5,2500,18.982428,6.787924,18.98±6.79,54.920000,49.767302,54.92±49.77,86.120000,34.580687,86.12±34.58,94.240000,23.303210,94.24±23.30
4,RGB-CLIP,4,Industrial,5,2500,51.553678,19.729308,51.55±19.73,88.960000,31.345038,88.96±31.35,97.880000,14.407937,97.88±14.41,98.720000,11.243316,98.72±11.24
5,RGB-CLIP,5,Pasture,5,2000,21.664493,8.016587,21.66±8.02,68.650000,46.403169,68.65±46.40,93.450000,24.746792,93.45±24.75,97.800000,14.672002,97.80±14.67
6,RGB-CLIP,6,PermanentCrop,5,2500,24.278727,9.642573,24.28±9.64,74.320000,43.695553,74.32±43.70,92.360000,26.569019,92.36±26.57,96.040000,19.505655,96.04±19.51
7,RGB-CLIP,7,Residential,5,3000,39.656292,14.092010,39.66±14.09,88.500000,31.907513,88.50±31.91,97.800000,14.670779,97.80±14.67,98.966667,10.114329,98.97±10.11
8,RGB-CLIP,8,River,5,2500,19.672361,5.176467,19.67±5.18,59.120000,49.171056,59.12±49.17,90.640000,29.132969,90.64±29.13,95.880000,19.879227,95.88±19.88
9,RGB-CLIP,9,SeaLake,5,3000,59.827572,27.746895,59.83±27.75,89.266667,30.958817,89.27±30.96,95.033333,21.729157,95.03±21.73,96.800000,17.602934,96.80±17.60


## Exported Files

Main CSV outputs:

- `eurosat_5fold_fold_metrics.csv`
- `eurosat_5fold_comparison.csv`
- `eurosat_5fold_paired_ttest.csv`
- `eurosat_5fold_per_class.csv`
- `eurosat_5fold_per_query.csv`
- `eurosat_5fold_split_manifest.csv`


In [15]:
sorted(str(path) for path in RESULTS_DIR.glob("*.csv"))

['/Users/tienesng06/Desktop/ACIVS_ThayBach/results/eurosat_5fold_cv/eurosat_5fold_comparison.csv',
 '/Users/tienesng06/Desktop/ACIVS_ThayBach/results/eurosat_5fold_cv/eurosat_5fold_fold_metrics.csv',
 '/Users/tienesng06/Desktop/ACIVS_ThayBach/results/eurosat_5fold_cv/eurosat_5fold_paired_ttest.csv',
 '/Users/tienesng06/Desktop/ACIVS_ThayBach/results/eurosat_5fold_cv/eurosat_5fold_per_class.csv',
 '/Users/tienesng06/Desktop/ACIVS_ThayBach/results/eurosat_5fold_cv/eurosat_5fold_per_query.csv',
 '/Users/tienesng06/Desktop/ACIVS_ThayBach/results/eurosat_5fold_cv/eurosat_5fold_split_manifest.csv']